In [ ]:
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
import ee
from PIL import Image

from tile_utils import draw_marker


# Paleta topográfica: valles/tierras bajas en verde, zonas medias en
# amarillo/marrón, cumbres en blanco/gris (referencia orientativa de rangos
# altitudinales para Colombia, ajustar 'max' si tu región es más alta/baja)
ELEVATION_PALETTE = [
    '004400',  # 0 – 364 m: Valles profundos, tierras bajas o planicies costeras
    '006600',  # 364 – 727 m: Tierras bajas / Bosques húmedos tropicales
    '38a800',  # 727 – 1,091 m: Transición baja / Inicio de piedemontes
    '73d216',  # 1,091 – 1,455 m: Zonas cafeteras o agrícolas de clima templado
    'b2d235',  # 1,455 – 1,818 m: Laderas de montaña media
    'fce94f',  # 1,818 – 2,182 m: Tierras medias altas / Bosques andinos
    'e9b96e',  # 2,182 – 2,545 m: Zonas altoandinas / Transición fría
    'c87d32',  # 2,545 – 2,909 m: Páramos bajos / Montaña alta
    '8f5902',  # 2,909 – 3,273 m: Páramos altos / Suelos rocosos fríos
    '5c3a21',  # 3,273 – 3,636 m: Superpáramo / Cumbres escarpadas
    '8b8b8b',  # 3,636 – 4,000+ m: Picos más altos / Gris roca alpino
]


def download_regional_elevation_map(lat, lon, region_type="state", out_prefix="regional_elevation",
                                     output_dir="../img/maps", dimensions=720,
                                     min_elevation=0, max_elevation=4000, show_marker=True):
    """
    Descarga un mapa de elevación (DEM, paleta topográfica) de la región
    administrativa (state o country) que contiene el punto dado.

    lat, lon: coordenadas del punto de referencia (define la región Y se
               marca en el mapa si show_marker=True)
    region_type: "state" (nivel 1, departamento/provincia) o "country" (nivel 0)
    dimensions: tamaño en pixeles del lado más largo de la imagen
    min_elevation, max_elevation: rango de la paleta de colores (metros)
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    """
    point = ee.Geometry.Point([lon, lat])

    if region_type == "country":
        feature = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).first()
        name_key = 'ADM0_NAME'
    else:
        feature = ee.FeatureCollection("FAO/GAUL/2015/level1").filterBounds(point).first()
        name_key = 'ADM1_NAME'

    region_name = feature.get(name_key).getInfo()
    geom = feature.geometry()
    print(f"[DEBUG] Zona detectada: {region_name}")

    image = ee.Image('USGS/SRTMGL1_003').select('elevation')
    clipped = image.clip(geom)

    vis_image = clipped.visualize(
        min=min_elevation,
        max=max_elevation,
        palette=ELEVATION_PALETTE,
    )

    thumbnail_url = vis_image.getThumbURL({
        'region': geom,
        'dimensions': dimensions,
        'format': 'png',
    })

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    resp = requests.get(thumbnail_url, timeout=30)
    resp.raise_for_status()
    canvas = Image.open(BytesIO(resp.content)).convert("RGBA")

    if show_marker:
        # Interpolación lineal dentro del bbox de la geometría (aproximación
        # razonable para referencia visual; a escala de state/país hay algo
        # de distorsión por la proyección, pero sirve para ubicar el punto).
        bounds_coords = geom.bounds().getInfo()["coordinates"][0]
        lons = [c[0] for c in bounds_coords]
        lats = [c[1] for c in bounds_coords]
        west, east = min(lons), max(lons)
        south, north = min(lats), max(lats)

        px = (lon - west) / (east - west) * canvas.width
        py = (north - lat) / (north - south) * canvas.height
        draw_marker(canvas, px, py)

    canvas.save(out_path)
    print(f"Imagen guardada en {out_path} ({canvas.width}x{canvas.height}px)")
    return out_path


if __name__ == "__main__":
    
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    
    lat_test = 7.4584221918243045
    lon_test = -73.222052853104

    download_regional_elevation_map(lat_test, lon_test, region_type="state")

[DEBUG] Zona detectada: Santander
Imagen guardada en ../img/maps/regional_elevation-v260805194634.png (607x720px)
